# Descriptions cleaning

In [12]:
from pathlib import Path

import pandas as pd

# Select the existing JSON or Parquet input; neither input file is modified.
INPUT_FORMAT = "json"
input_paths = {
    "json": Path("data/processed/books_clean_balanced.json"),
    "parquet": Path("data/processed/books_clean_balanced.parquet"),
}
output_json_path = Path("data/processed/BALANCED_CLEANED.json")
output_parquet_path = Path("data/processed/BALANCED_CLEANED.parquet")

if INPUT_FORMAT not in input_paths:
    raise ValueError(f"Unsupported input format {INPUT_FORMAT!r}; choose 'json' or 'parquet'.")
input_path = input_paths[INPUT_FORMAT]
books = pd.read_json(input_path) if INPUT_FORMAT == "json" else pd.read_parquet(input_path)
books_updated = books.copy()


def _description_text(value):
    if isinstance(value, dict):
        value = value.get("value", "")
    if isinstance(value, list):
        return "\n".join(str(item) for item in value if item is not None)
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def save_balanced_descriptions(frame):
    books_to_save = frame.drop(columns=["word_count"], errors="ignore")
    books_to_save.to_json(output_json_path, orient="records", indent=2, force_ascii=False)
    books_to_save.to_parquet(output_parquet_path, engine="pyarrow", index=False)
    print(f"[save] {len(books_to_save):,} books -> {output_json_path} and {output_parquet_path}")


print(f"[input] loaded {len(books_updated):,} books from {input_path}")
print(f"[output] {output_json_path} and {output_parquet_path}")
save_balanced_descriptions(books_updated)


[input] loaded 34,844 books from data/processed/books_clean_balanced.json
[output] data/processed/BALANCED_CLEANED.json and data/processed/BALANCED_CLEANED.parquet
[save] 34,844 books -> data/processed/BALANCED_CLEANED.json and data/processed/BALANCED_CLEANED.parquet


In [13]:
import re
import time

import requests

# Use the source description unless a cleaned description is already present.
description_column = "desc_cleaned" if "desc_cleaned" in books_updated.columns else "description"
books_updated["desc_cleaned"] = books_updated[description_column].map(_description_text)
short_indices = books_updated.index[books_updated["desc_cleaned"].str.split().str.len() < 10]
print(f"[lookup] {len(short_indices):,} descriptions have fewer than 10 words")

session = requests.Session()
session.headers.update({"User-Agent": "BookDescriptionCleaner/1.0"})
checked_editions = 0
replaced = 0
request_errors = 0
missing_work_keys = 0


def _openlibrary_json(url, params=None):
    global request_errors
    for attempt in range(1, 4):
        try:
            response = session.get(url, params=params, timeout=(5, 20))
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict):
                raise ValueError("Expected a JSON object")
            return payload
        except (requests.RequestException, ValueError) as exc:
            status = exc.response.status_code if isinstance(exc, requests.HTTPError) and exc.response is not None else None
            retryable = isinstance(exc, (requests.Timeout, requests.ConnectionError)) or status in (429, 500, 502, 503, 504)
            if not retryable or attempt == 3:
                request_errors += 1
                raise RuntimeError(f"Open Library lookup failed for {url}: {exc}. No books were filtered; rerun this cell to resume in memory.") from exc
            delay = 2 ** (attempt - 1)
            print(f"[retry {attempt}/2] {url}: {exc}; waiting {delay}s", flush=True)
            time.sleep(delay)
        finally:
            time.sleep(0.1)


def _book_path(value):
    if not isinstance(value, str):
        return None
    key = value.strip()
    if re.fullmatch(r"OL[0-9]+M", key):
        return f"/books/{key}"
    if re.fullmatch(r"/books/OL[0-9]+M", key):
        return key
    return None


def _edition_description(edition, seen, fetch_details=False):
    global checked_editions
    key = _book_path(edition.get("key"))
    if key is not None:
        if key in seen:
            return ""
        seen.add(key)
    text = _description_text(edition.get("description", ""))
    if len(text.split()) >= 10:
        checked_editions += 1
        return text
    if key is None or not fetch_details:
        return ""
    checked_editions += 1
    details = _openlibrary_json(f"https://openlibrary.org{key}.json")
    text = _description_text(details.get("description", ""))
    return text if len(text.split()) >= 10 else ""


for processed, index in enumerate(short_indices, 1):
    row = books_updated.loc[index]
    keys = row.get("edition_key")
    if isinstance(keys, str):
        keys = [keys]
    elif not isinstance(keys, (list, tuple)):
        keys = []
    seen = set()
    replacement = ""
    for key in keys:
        replacement = _edition_description({"key": key}, seen, fetch_details=True)
        if replacement:
            break

    if not replacement:
        work_key = row.get("work_key")
        if not isinstance(work_key, str) or not re.fullmatch(r"/works/OL[0-9]+W", work_key.strip()):
            missing_work_keys += 1
        else:
            offset = 0
            page_size = 100
            while True:
                url = f"https://openlibrary.org{work_key.strip()}/editions.json"
                page = _openlibrary_json(url, params={"limit": page_size, "offset": offset})
                entries = page.get("entries")
                if not isinstance(entries, list):
                    raise ValueError(f"Open Library returned a missing or invalid editions list for {url}; no books were filtered.")
                for edition in entries:
                    if not isinstance(edition, dict):
                        raise ValueError(f"Open Library returned an invalid edition entry for {url}; no books were filtered.")
                    replacement = _edition_description(edition, seen)
                    if replacement:
                        break
                if replacement or len(entries) < page_size:
                    break
                offset += page_size

    if replacement:
        books_updated.at[index, "desc_cleaned"] = replacement
        replaced += 1
    if processed % 10 == 0 or processed == len(short_indices):
        print(f"[progress] {processed:,}/{len(short_indices):,} books; {checked_editions:,} editions checked; {replaced:,} replaced", flush=True)

word_count = books_updated["desc_cleaned"].str.split().str.len()
keep = word_count >= 10
removed = int((~keep).sum())
books_updated = books_updated.loc[keep].copy().reset_index(drop=True)
print(f"[lookup] checked {checked_editions:,} editions; replaced {replaced:,} descriptions")
print(f"[lookup] {request_errors:,} request/response errors; {missing_work_keys:,} books without a valid work key")
print(f"[filter] removed {removed:,} books still under 10 words; {len(books_updated):,} remain")
save_balanced_descriptions(books_updated)


[lookup] 436 descriptions have fewer than 10 words
[retry 1/2] https://openlibrary.org/books/OL28099720M.json: HTTPSConnectionPool(host='openlibrary.org', port=443): Read timed out. (read timeout=5); waiting 1s
[progress] 10/436 books; 17 editions checked; 8 replaced
[progress] 20/436 books; 34 editions checked; 18 replaced
[progress] 30/436 books; 49 editions checked; 26 replaced
[progress] 40/436 books; 61 editions checked; 34 replaced
[progress] 50/436 books; 74 editions checked; 41 replaced
[progress] 60/436 books; 89 editions checked; 46 replaced
[progress] 70/436 books; 103 editions checked; 54 replaced
[progress] 80/436 books; 119 editions checked; 62 replaced
[progress] 90/436 books; 129 editions checked; 63 replaced
[progress] 100/436 books; 142 editions checked; 71 replaced
[progress] 110/436 books; 156 editions checked; 76 replaced
[progress] 120/436 books; 170 editions checked; 81 replaced
[progress] 130/436 books; 183 editions checked; 85 replaced
[progress] 140/436 books;

### Description-cleaning pipeline

Run the setup and enrichment cells above first, then run these cleaning cells in order. Each stage saves to the designated output files.


In [14]:
import html
import re
import unicodedata

import ftfy
import pandas as pd

if "books_updated" not in globals():
    raise NameError("Run the setup and enrichment cells first to load and prepare books_updated.")
if "desc_cleaned" not in books_updated.columns:
    books_updated["desc_cleaned"] = books_updated["description"].fillna("")

def _description_text(value):
    if value is None:
        return ""
    if isinstance(value, dict):
        value = value.get("value", "")
    if isinstance(value, list):
        return "\n".join(str(item) for item in value if item is not None)
    if pd.isna(value):
        return ""
    return str(value)

books_updated = books_updated.copy()
books_updated["desc_cleaned"] = books_updated["desc_cleaned"].map(_description_text)
_initial_descriptions = books_updated["desc_cleaned"].copy()

def _run_cleaning_step(label, transform):
    before = books_updated["desc_cleaned"].copy()
    after = transform(before).fillna("").map(_description_text)
    changed = before.ne(after)
    books_updated["desc_cleaned"] = after
    print(f"[clean] {label}: {int(changed.sum()):,} descriptions changed")
    for index in changed[changed].index[:2]:
        title = books_updated.at[index, "title"] if "title" in books_updated else index
        old = before.at[index].replace("\n", " ")[:180]
        new = after.at[index].replace("\n", " ")[:180]
        print(f"  - {title}: {old!r} -> {new!r}")

def _tidy_whitespace(series):
    return (
        series.str.replace(r"[ \t]{2,}", " ", regex=True)
        .str.replace(r"\n[ \t]+", "\n", regex=True)
        .str.replace(r"\n{3,}", "\n\n", regex=True)
        .str.strip()
    )

print(f"Starting description cleaning: {len(books_updated):,} books")
print(f"Empty descriptions: {(books_updated['desc_cleaned'].str.strip() == '').sum():,}")
print(f"Descriptions under 10 words: {(books_updated['desc_cleaned'].str.split().str.len() < 10).sum():,}")

save_balanced_descriptions(books_updated)


Starting description cleaning: 34,618 books
Empty descriptions: 0
Descriptions under 10 words: 0
[save] 34,618 books -> data/processed/BALANCED_CLEANED.json and data/processed/BALANCED_CLEANED.parquet


In [15]:
# Convert markup to readable text and remove catalog, website, and source clutter.
def _clean_markup(series):
    series = series.map(lambda text: unicodedata.normalize("NFC", ftfy.fix_text(text)))
    series = series.str.replace(
        r"(?i)<\s*(?:br\s*/?|/?(?:p|div|li|h[1-6]|ul|ol|blockquote))\b[^>]*>",
        "\n", regex=True
    )
    series = series.str.replace(r"(?is)\(?\s*<a\s+[^>]*>\s*source\s*</a>\s*\)?", "", regex=True)
    series = series.str.replace(r"<[^>]+>", "", regex=True).map(html.unescape)
    series = series.str.replace(r"\[([^\]]+)\]\s*\(\s*https?://[^)]+\)", r"\1", regex=True)
    series = series.str.replace(r"\[([^\]]+)\]\[\d+\]", r"\1", regex=True)
    series = series.str.replace(r"\[([^\]]+)\]\([^)]+\)", r"\1", regex=True)
    series = series.str.replace(r"(?m)^\s*#{1,6}\s+", "", regex=True)
    series = series.str.replace(r"\*{1,3}([^*]+)\*{1,3}", r"\1", regex=True)
    series = series.str.replace(r"_{1,2}([^_]+)_{1,2}", r"\1", regex=True)
    return _tidy_whitespace(series)

_run_cleaning_step("HTML, Markdown, and whitespace", _clean_markup)

def _extract_synopsis(text):
    match = re.search(r"(?im)^\s*Synopsis\s*\n", text)
    return text[match.end():] if match else text

def _extract_library_summary(text):
    if re.search(r"Librarian's View.*?Summary\s*\n", text, flags=re.I | re.S):
        return re.split(r"Summary\s*\n", text, flags=re.I)[-1]
    return text

_run_cleaning_step("Extract synopsis and library summary", lambda s: s.map(_extract_synopsis).map(_extract_library_summary))

catalog_cutoff = re.compile(
    r"\n\s*(?:-{3,}\s*\n|"
    r"(?:Also contained in|Contains|See also|Followed by|Preceded by|Contents)\s*:|"
    r"\(?\[source\](?:\[\d+\]|\([^)]+\))?\)?|"
    r"\[\d+\]:\s*https?://|"
    r"Details\s*\n\s*(?:First Published|Publisher|ISBN)\s*:|"
    r"Excerpt\s*\n|Dedication\s*\n|"
    r"Privacy Policy\s*\|\s*Cookie|©.*All Rights Reserved\.?|"
    r"ABOUT AUTHOR\s*:)",
    flags=re.I,
)

def _cut_catalog_sections(text):
    return catalog_cutoff.split(text, maxsplit=1)[0]

_run_cleaning_step("Catalog sections, footers, and author notes", lambda s: s.map(_cut_catalog_sections))

source_patterns = [
    (r"(?is)\n\s*(?:Source\s*:\s*)?(?:\[?Wikipedia\]?(?:\([^)]+\)|\[\d+\])?.*|From Wikipedia, the free encyclopedia\.?.*)$", ""),
    (r"(?i)^\s*[\(\[]?\s*(?:From\s+)?Wikipedia(?:\s*\([^)]+\))?\s*[:\)\]]?\s*", ""),
    (r"(?i)\s*\([^)]*(?:wikipedia\.org|wikipedia)[^)]*\)\.?", ""),
    (r"(?i)\s*(?:I got this from Wikipedia|See (?:more|the Wikipedia entry)[^.]*wikipedia[^.]*|More on Wikipedia[^.]*|Read more at\s*https?://[^\s]*wikipedia[^\s]*|From English Wikipedia)\.?\s*$", ""),
    (r"(?i)(?:\s*[-–—]{1,2}\s*|\n\s*[-–—]?\s*)(?:modified slightly from\s+|taken from\s+|from\s+|main source\s*:?\s*|source\s*:?\s*)?\*?(?:www\.)?(?:en\.)?wikipedia(?:\.com|\.org)?\*?\.?", ""),
    (r"(?i)\s*The description at this Wikipedia entry is pretty close\..*$", ""),
    (r"(?i)\s*<a\s+[^>]*href=[\"']?https?://[^\"'>]*wikipedia\.org[^\"'>]*[\"']?[^>]*>\s*Wikipedia\s*</a>", ""),
    (r"(?i)\s*\([^)]*(?:copied\s+from\s+)?goodreads(?:\.com)?[^)]*\)", ""),
    (r"(?is)\n\s*GOODREADS Review\s*:[\s\S]*$", ""),
    (r"(?im)^\s*\(?\s*(?:From\s+(?:book\s+description\s+on\s+)?Goodreads|As\s+described\s+on\s+goodreads|Goodreads\s+Review)\s*\)?\s*:?\s*", ""),
    (r"(?i)\s*(?:[-–—]{1,2}\s*|\(\s*)(?:This description comes from (?:the )?publisher|Provided by (?:the )?publisher|Container|Publisher(?:'s)? (?:description|website)|Dust jacket|Back cover|Amazon\.com)\s*\.?\)?\s*$", ""),
    (r"(?i)^\s*From the back cover\s*:?[ \t]*", ""),
    (r"(?i)^\s*From\s+[a-z0-9-]+\s+dot\s+(?:com|net|org|uk)\s*:\s*", ""),
    (r"(?i)^\s*Download\s+(?:This\s+Book|on)\s*[-→➞]*\s*https?://\S+\s*", ""),
    (r"(?i)\s*https?://[a-z.]*wikipedia\.org/[^\s)\]]+", ""),
    (r"(?i)^\s*(?:This is a duplicate\.\s*Please update your lists\.\s*)?See(?:\s+work)?\s*:?\s*<?https?://[^\s>]+>?\s*\.?\s*$", ""),
    (r"(?i)\s*(?:\[Also at archive\.org[^\]]*\]|\(\s*(?:14-day borrow|source)[^)]*https?://[^)]*\)|\n?\s*[-–—.]*\s*(?:Source|See more at|See work|Part of an Open Library list)\s*:?\s*(?:https?://|www\.)\S+)\s*\.?", ""),
    (r"(?i)^\s*\(?source\)?\s*$", ""),
    (r"(?i)\(?\s*<?(?:https?://|www\.)[^\s<>\)\]]+>?\s*\)?", ""),
    (r"\[\d+\]", ""),
]

def _clean_sources(series):
    for pattern, replacement in source_patterns:
        series = series.str.replace(pattern, replacement, regex=True)
    return _tidy_whitespace(series)

_run_cleaning_step("Wikipedia, Goodreads, publisher credits, citations, and URLs", _clean_sources)

save_balanced_descriptions(books_updated)


[clean] HTML, Markdown, and whitespace: 14,735 descriptions changed
  - It Ends With Us: 'Lily hasn’t always had it easy, but that’s never stopped her from working hard for the life she wants. She’s come a long way from the small town where she grew up—she graduated fro' -> "Lily hasn't always had it easy, but that's never stopped her from working hard for the life she wants. She's come a long way from the small town where she grew up—she graduated fro"
  - Harry Potter and the Philosopher's Stone: "Turning the envelope over, his hand trembling, Harry saw a purple wax seal bearing a coat of arms; a lion, an eagle, a badger and a snake surrounding a large letter 'H'.\r \r HARRY P" -> "Turning the envelope over, his hand trembling, Harry saw a purple wax seal bearing a coat of arms; a lion, an eagle, a badger and a snake surrounding a large letter 'H'.  HARRY POT"
[clean] Extract synopsis and library summary: 2 descriptions changed
  - Fatherland: "Fatherland is set in a world that almos

In [16]:
# Remove non-plot metadata, promotional copy, lists, and description formatting noise.
non_plot_sentence = re.compile(
    r"(?i)(?:"
    r"\b(?:won|winner of|awarded|nominated for|received nominations for|shortlisted for|longlisted for|finalist for|recognized as)\b[^.!?]*\b(?:award|prize|medal|honor|honour|accolades)\b|"
    r"\b(?:became|was|is)\s+(?:a|an|his|her|their|the)?\s*(?:instant\s+|international\s+|worldwide\s+|runaway\s+|first\s+|number-one\s+|#1\s+)?best[\s-]?sellers?\b|"
    r"\b(?:made|stayed on|reached|topped)\s+(?:the\s+)?(?:new york times|usa today|sunday times)\s+best[\s-]?seller\b|"
    r"\blisted at number\s+\d+\s+on the\b|"
    r"\b(?:received|garnered|met with)\s+(?:critical\s+acclaim|positive\s+reviews|mixed\s+reviews|widespread\s+acclaim|praise)\b|"
    r"\b(?:it|the\s+(?:novel|book))\s+was\s+(?:widely\s+)?praised\s+for\b|"
    r"\b(?:was\s+)?(?:first\s+)?published\s+(?:in\s+hardcover\s+|in\s+paperback\s+|posthumously\s+)?(?:on|in|by)\b|"
    r"\bfeaturing\s+a\s+cover\s+designed\s+by\b|"
    r"\b(?:the\s+(?:novel|book)|it)\s+was\s+(?:written|completed)\s+(?:over|in|between|during)\b|"
    r"\b(?:it\s+was\s+)?translated\s+into\s+(?:english|\d+|over\s+\d+)\b|"
    r"\b(?:was\s+)?(?:adapted|made|turned)\s+into\s+a\s+(?:feature\s+)?(?:film|movie|television|tv|stage|miniseries)\b|"
    r"\bis\s+(?:a|an)\s+\d{4}\s+(?:[\w-]+\s+){0,3}(?:novel|book|novella|story|collection|memoir|biography)\s+(?:written\s+)?by\b|"
    r"\b(?:new york times|nyt|usa today|publishers weekly|kirkus|booklist)\b[^.!?]{0,100}\b(?:review|bestseller|best-seller|author)\b"
    r")"
)
plot_context = re.compile(
    r"(?i)\b(?:tells the story|story follows|novel follows|book follows|follows|centers on|set in|plot|protagonist|character|about the)\b"
)

def _clean_non_plot_sentences(text):
    cleaned_paragraphs = []
    for paragraph in text.split("\n\n"):
        sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z“\"'*])", paragraph)
        kept = [
            sentence for sentence in sentences
            if not (non_plot_sentence.search(sentence) and not plot_context.search(sentence))
        ]
        if kept:
            cleaned_paragraphs.append(" ".join(kept))
    return "\n\n".join(cleaned_paragraphs)

promo_block = re.compile(
    r"(?i)(?:"
    r"\b(?:new york times|usa today|sunday times|international|national|#\d+)\b[^.\n]{0,80}\bbest[\s-]?seller\b|"
    r"^(?:an?\s+)?(?:instant\s+)?best[\s-]?seller\b|"
    r"\b(?:won|winner of|nominated for|shortlisted for|longlisted for|finalist for)\b[^.\n]{0,100}\b(?:award|prize|medal)\b|"
    r"^(?:winner|finalist|nominee|shortlisted|longlisted)\s+(?:of|for)\b|"
    r"^as seen on\b|^from\s+(?:the\s+)?(?:#\d+\s+)?(?:new york times|usa today|award-winning|bestselling)\s+author\b|"
    r"^international bestseller\b|^“[^”]+”\s*[—–-]\s*[A-Z]"
    r")"
)

def _clean_promotional_blocks(text):
    paragraphs = [paragraph.strip() for paragraph in text.split("\n\n") if paragraph.strip()]
    kept = []
    for paragraph in paragraphs:
        lines = [line for line in paragraph.splitlines() if not promo_block.search(line.strip())]
        paragraph = "\n".join(lines).strip()
        if paragraph and not promo_block.search(paragraph):
            kept.append(paragraph)
    return "\n\n".join(kept)

def _clean_goodreads_and_lists(text):
    text = re.sub(r"(?is)^[\s\S]{0,250}?\b\d\.\d{2}\s*·\s*Rating details\s*·[^\n]+\n+", "", text)
    text = re.sub(r"(?i)\n\s*These can be read in the order published.*?https?://\S+\s*", "", text)
    text = re.sub(r"(?is)\n*\btable of contents\b[\s\S]*$", "", text)
    text = re.sub(r"(?i)\s*Suggested level\s*:[^.\n]+\.?\s*$", "", text)
    text = re.sub(r"(?im)\n*\s*strong text author:.*$", "", text)
    text = re.sub(r"(?i)\s*\(\s*Amazon\.com\s+Review\s*--[^)]+\)\s*", " ", text)
    text = re.sub(r"(?i)\s*\(\s*(?:Ages?|Grades?|Gr\.?|Preschool|Pre-K|Kindergarten)\s*[^)]*\)\s*\*?", " ", text)
    text = re.sub(r"(?i)\s*[-–—]{1,2}\s*This text refers to[^.\n]+\.?", "", text)
    text = re.sub(r"(?i)\s*\(\s*(?:copied\s+from\s+)?goodreads(?:\.com)?\s*\)", "", text)
    text = re.sub(r"(?i)\s*\([^()]{3,120}\b(?:bestselling\s+author|author\s+of|award-winning\s+author|creator\s+of|USA\s+TODAY|New\s+York\s+Times|NYT|Washington\s+Post|Publishers\s+Weekly|Kirkus|Booklist|Entertainment\s+Weekly|People|NPR|The\s+Guardian|Sunday\s+Times)[^()]*\)", "", text)
    return text

def _clean_series_and_catalog_noise(text):
    text = re.sub(r"(?i)\n\s*[^\n:]+:\s*(?:\n\s*[^\n]+\([^)]+,\s*#\d+\)[ \t]*)+$", "", text)
    text = re.sub(r"\n\s*\n[A-Z][^\n:]{2,50}:\s*(?:\n[^\n.!?]{2,60}){2,}\s*$", "", text)
    text = re.sub(r"^[A-Z][^\n]{2,50}#\d+\s*\n+", "", text)
    text = re.sub(r"(?i)\n[^\n:]+:\s*\n[^\n]+\([^)]+,\s*#\d+(?:\.\d+)?\)[\s\S]*$", "", text)
    text = re.sub(r"(?i)\s*\(?\s*(?:Book|Volume|Vol\.?|Part)\s+\d+\s+of\s+[^)]*\)?\s*$", "", text)
    text = re.sub(r"(?i)\s*\(?\s*(?:Part of the|Sequel to|Prequel to|Cycle\b.*\btome)\s+[^)]*\)?\s*$", "", text)
    return text

def _clean_formatting(text):
    text = re.sub(r"(?i)^\s*\(Gr\.\s*[\d-]+\s*Age\s*[\d-]+\)\s*", "", text)
    text = re.sub(r"(?i)\s+[a-z]{2,4}\s+\d{2}-\d{2}-\d{2,4}\s*$", "", text)
    text = re.sub(r"^\"\s*(.*?)\s*(?:\"\s*[-–—]{1,2}|[-–—]{1,2})\s*$", r"\1", text)
    text = re.sub(r"(?i)\s*[-–—]{1,4}\s*(?:Cover|Jacket\s+flap|Google\s+Books|originally\s+published\s+as\s+[^\n]+|(?:p\.?|page)\s*4\s+of\s+cover)\.?\s*$", "", text)
    text = re.sub(r"\s*\[\d+\]", "", text)
    text = re.sub(r"([.!?])\s*>\s*([A-Z])", r"\1 \2", text)
    text = re.sub(r"([a-z]{2,}[!?])([A-Z][a-z])", r"\1 \2", text)
    text = re.sub(r"([a-z]{2,}\.)([A-Z][a-z])", r"\1 \2", text)
    text = re.sub(r"([a-z]{3,})([A-Z][a-z]{2,})", r"\1 \2", text)
    text = re.sub(r"^\s*(?:\.\s*){3,}", "", text)
    text = re.sub(r"\s*#+\s*$", "", text)
    text = re.sub(r"^\[(.*)\]$", r"\1", text.strip())
    text = re.sub(r'^"\s*(.*?)\s*"?$', r"\1", text.strip())
    text = re.sub(r'([”\"]|[”\"])\s*\([^()]{3,120}\b(?:bestselling\s+author|author\s+of|USA\s+TODAY|New\s+York\s+Times|NYT|Publishers\s+Weekly|Kirkus|Booklist)[^()]*\)', r"\1", text, flags=re.I)
    text = re.sub(r"(?i)\b[^.!?]+\s+by\s+[A-Z][a-z]+\s+[A-Z][a-z]+\.?\s*$", "", text) if len(text.split()) <= 12 else text
    paragraphs = []
    seen = set()
    for paragraph in text.split("\n\n"):
        normalized = paragraph.strip()
        if not normalized:
            continue
        if normalized not in seen:
            seen.add(normalized)
            paragraphs.append(normalized)
    return "\n\n".join(paragraphs)

_run_cleaning_step("Promotional/metadata sentences and review quotes", lambda s: s.map(_clean_non_plot_sentences).map(_clean_promotional_blocks))
_run_cleaning_step("Goodreads fragments, age labels, series/catalog lists", lambda s: s.map(_clean_goodreads_and_lists).map(_clean_series_and_catalog_noise))
_run_cleaning_step("Formatting, duplicate paragraphs, and spacing", lambda s: s.map(_clean_formatting).pipe(_tidy_whitespace))

save_balanced_descriptions(books_updated)


[clean] Promotional/metadata sentences and review quotes: 4,374 descriptions changed
  - It Starts with Us: 'Before It Ends with Us, it started with Atlas. Colleen Hoover tells fan favorite Atlas\'s side of the story and shares what comes next in this long-anticipated sequel to the "glorio' -> 'Before It Ends with Us, it started with Atlas.  Lily and her ex-husband, Ryle, have just settled into a civil coparenting rhythm when she suddenly bumps into her first love, Atlas,'
  - A Game of Thrones: 'A Game of Thrones is the inaugural novel in A Song of Ice and Fire, an epic series of fantasy novels crafted by the American author George R. R. Martin. Published on August 1, 1996' -> 'A Game of Thrones is the inaugural novel in A Song of Ice and Fire, an epic series of fantasy novels crafted by the American author George R. R. Martin.  The story unfolds through '
[clean] Goodreads fragments, age labels, series/catalog lists: 918 descriptions changed
  - It Starts with Us: 'Before It Ends wit

In [17]:
from pathlib import Path

# Remove records that are metadata-only, title duplicates, or table-of-contents fragments.
descriptions = books_updated["desc_cleaned"].fillna("").astype(str)
word_count = descriptions.str.split().str.len()
short = word_count <= 12
short_noise = descriptions.str.contains(
    r"(?i)(?:\b\d+\s+volume\b.*\bcm\b|"
    r"^(?:Book|Volume|Vol\.?|Part)\s+\d+\s+of\b|"
    r"^(?:Part of the|Sequel to|Prequel to|Cycle\b.*\btome|other title\s*:)|"
    r"\bmissing pages\b|"
    r"^(?:very funny|really good|it was ok|i hear this book|its trilling|very interesting|this book is very scary)\b|"
    r"\b(?:\d+\s+volume|\d+\s*v\.|\d+\s*p\.|unnumbered pages|large print)\b.*\bcm\b|"
    r"\b(?:book|series)\s*#?\s*\d+\b|"
    r"^(?:first|second|third|fourth|fifth|book\s+\d+)\s+(?:book\s+)?in the\b|"
    r"^(?:novel,\s*)?sequel to\b|"
    r"^(?:first published|reprint of|shortlisted for|index of proper names|complete in one volume|review in the)\b|"
    r"\bopposite pages\.?$|(?:is wonderful!|not worth the bother)|\d+\s*yrs\b)",
    na=False,
) & short

normalized_desc = descriptions.str.lower().str.replace(r"[^\w\s]", "", regex=True).str.strip()
normalized_title = books_updated.get("title", pd.Series("", index=books_updated.index)).fillna("").astype(str).str.lower().str.replace(r"[^\w\s]", "", regex=True).str.strip()
same_as_title = (normalized_desc == normalized_title) & short

def _looks_like_table_of_contents(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) < 6:
        return False
    if sum(line.endswith("-") for line in lines) >= 2:
        return False
    if sum(line[0].islower() for line in lines) / len(lines) > 0.35:
        return False
    average_length = sum(len(line) for line in lines) / len(lines)
    sentence_endings = sum(bool(re.search(r"[.!?][\"”']?$", line)) for line in lines)
    return average_length < 55 and sentence_endings / len(lines) < 0.2

contents_only = descriptions.str.match(r"(?i)^\s*(?:Contains|Also contained in|Contents)\s*:", na=False)
table_of_contents = descriptions.map(_looks_like_table_of_contents)
too_short = word_count < 10
keep = ~(short_noise | same_as_title | contents_only | table_of_contents | too_short)
removed = int((~keep).sum())
print("[filter] short noise:", int(short_noise.sum()))
print("[filter] exact title duplicates:", int(same_as_title.sum()))
print("[filter] contents/TOC entries:", int((contents_only | table_of_contents).sum()))
print("[filter] fewer than 10 words after cleaning:", int(too_short.sum()))
print(f"[filter] removed {removed:,} of {len(books_updated):,} rows")

retained_indices = books_updated.index[keep]
changed_total = int(_initial_descriptions.loc[retained_indices].ne(books_updated.loc[retained_indices, "desc_cleaned"]).sum())
books_updated = books_updated.loc[keep].reset_index(drop=True)
print(f"[summary] {changed_total:,} retained descriptions changed; {len(books_updated):,} books remain")

from lingua import Language, LanguageDetectorBuilder

noisy_languages = [
    Language.LATIN, Language.ESPERANTO, Language.SOMALI, Language.SWAHILI, Language.TSWANA,
    Language.SOTHO, Language.XHOSA, Language.ZULU, Language.TSONGA, Language.YORUBA,
    Language.SHONA, Language.GANDA, Language.MAORI, Language.TAGALOG, Language.WELSH,
    Language.IRISH, Language.AFRIKAANS, Language.BASQUE, Language.ALBANIAN,
    Language.NYNORSK, Language.CATALAN,
]
description_detector = LanguageDetectorBuilder.from_all_languages_without(*noisy_languages).build()

def _classify_description_language(text):
    if not text.strip():
        return (None, None, "empty")
    if len(re.findall(r"[A-Za-z]", text)) < 20:
        return (None, None, "uncertain_short")
    language = description_detector.detect_language_of(text[:2000])
    if language is None:
        return (None, None, "uncertain")
    iso_code = language.iso_code_639_1
    language_code = iso_code.name.lower() if iso_code is not None else language.name.lower()
    is_english = language == Language.ENGLISH
    return (language_code, is_english, "english" if is_english else "non_english")

language_results = books_updated["desc_cleaned"].map(_classify_description_language)
books_updated[["desc_language", "desc_is_english", "desc_language_status"]] = pd.DataFrame(
    language_results.tolist(), index=books_updated.index, columns=["desc_language", "desc_is_english", "desc_language_status"]
)
print("[language] English:", int((books_updated["desc_language_status"] == "english").sum()))
print("[language] non-English flagged (retained):", int((books_updated["desc_language_status"] == "non_english").sum()))
uncertain_statuses = ["uncertain", "uncertain_short", "empty"]
uncertain_count = books_updated["desc_language_status"].isin(uncertain_statuses).sum()
print("[language] uncertain/empty (retained):", int(uncertain_count))


save_balanced_descriptions(books_updated)


[filter] short noise: 3
[filter] exact title duplicates: 2
[filter] contents/TOC entries: 311
[filter] fewer than 10 words after cleaning: 282
[filter] removed 588 of 34,618 rows
[summary] 18,786 retained descriptions changed; 34,030 books remain
[language] English: 34001
[language] non-English flagged (retained): 29
[language] uncertain/empty (retained): 0
[save] 34,030 books -> data/processed/BALANCED_CLEANED.json and data/processed/BALANCED_CLEANED.parquet
